# 11 — Historical Analysis Through the Graph

**Objective**: run the compiled LangGraph workflow across several simulated weeks and watch week-over-week trend classification (Section 14) and multi-sprint blocker persistence (Section 12) emerge from the orchestrated pipeline itself — `persist_snapshot` writing real history, `retrieve_historical_memory` reading it back — rather than from hand-assembled `ProjectSnapshot`s the way notebook 09 built them.

**Dependencies**: `10_langgraph_workflow.ipynb`.

**A real architectural seam, surfaced honestly**: `fetch_financial_data` always asks the source for its CURRENT latest reporting period (no `as_of` override) — correct for a live weekly pipeline run, which always wants truly-latest numbers, but it means invoking the compiled graph at a *past* `requested_at` still pulls the *current* financial period, not the period that was actually current back then. Part A below (running forward from today) isn't affected by this. Part B (replaying a real past improvement/deterioration) needs a small period-pinning wrapper around the financial source to work around it — built and explained inline, not hidden.

In [1]:
import os
import sys
import shutil
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from datetime import date, datetime, timezone

from src.connectors.jira_client import build_default_jira_client
from src.connectors.financial_client import CSVFinancialDataSource, FinancialDataSource
from src.services import project_unifier, trend_engine
from src.services.memory_store import FileMemoryStore, MemoryScope
from src.graph.nodes import NodeDeps
from src.graph.workflow import build_graph

NOTEBOOK_SNAPSHOT_DIR = PROJECT_ROOT / "data/snapshots"
shutil.rmtree(NOTEBOOK_SNAPSHOT_DIR, ignore_errors=True)

mapping = project_unifier.load_project_mapping()
scope = MemoryScope(mapping.organization_id, mapping.portfolio_id)
memory_store = FileMemoryStore(base_dir=NOTEBOOK_SNAPSHOT_DIR)

print("Ready.")

Ready.


## Part A: running the live graph forward, week over week

Nothing in the underlying source systems changes between these three runs — this is a legitimate state for a stalled data feed to be in, and the pipeline should say so, not invent movement that isn't there.

In [2]:
deps = NodeDeps(
    jira_client=build_default_jira_client(),
    financial_source=CSVFinancialDataSource(),
    memory_store=memory_store,
    mapping=mapping,
)
graph = build_graph(deps)

forward_weeks = [datetime(2026, 9, 1, tzinfo=timezone.utc), datetime(2026, 9, 8, tzinfo=timezone.utc), datetime(2026, 9, 15, tzinfo=timezone.utc)]
for requested_at in forward_weeks:
    graph.invoke({"user_question": "Why is Phoenix Platform Modernization at risk?", "request_id": str(requested_at), "requested_at": requested_at})

phx_history = memory_store.get_snapshots(scope, "PROJECT-10001")
print(f"{len(phx_history)} PHX snapshots persisted across 3 weekly graph runs\n")
for s in phx_history:
    print(f"  {s.snapshot_date}: completion={s.sprint_completion_pct}%  blocked={s.blocked_issues}  rag={s.rag_status.value}")

print()
print("Week 1 -> 2:", trend_engine.classify_trend(phx_history[0], phx_history[1]).value)
print("Week 2 -> 3:", trend_engine.classify_trend(phx_history[1], phx_history[2]).value)

{"request_id": "2026-09-01 00:00:00+00:00", "timestamp": "2026-09-03T03:22:58.865112+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "2026-09-01 00:00:00+00:00", "timestamp": "2026-09-03T03:22:58.865259+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.01}
{"request_id": "2026-09-01 00:00:00+00:00", "timestamp": "2026-09-03T03:22:58.865756+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "2026-09-01 00:00:00+00:00", "timestamp": "2026-09-03T03:22:58.869681+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 37, "sprint_count": 3, "partial_failure": false}
{"request_id": "2026-09-01 00:00:00+00:00", "timestamp": "2026-09-03T03:22:58.869759+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 3.96}
{"request_id": "2026-09-01 00:00:00+00:00", "timestamp": "2026-09-03T03:22:58.870795+00:00", "event": "graph_node_start", "node": "validate_delivery_data

Correctly `STABLE` both times — the source data hasn't moved, so neither should the trend classification. The blocker-age numbers inside each snapshot's underlying issues did tick up day to day (visible in notebook 10's evidence output), but nothing crossed a `trend_thresholds` boundary, which is exactly what should happen for a genuinely unchanged week.

## Part B: replaying a real past improvement and deterioration, through the graph

PHX's real Sprint 2 -> Sprint 3 window (2026-01-19 to 2026-02-15) showed a genuine completion jump (25% -> 80%); TITAN's real Sprint 2 -> Sprint 3 showed a genuine drop (58.2% -> 40.6%) — the same two examples notebook 09 built by hand. This time the compiled graph runs it, using a small wrapper to pin the financial source to a specific historical period (see this notebook's intro cell for why that's needed).

In [3]:
class PeriodPinnedFinancialSource(FinancialDataSource):
    """Wraps a real FinancialDataSource and pins get_latest_reporting_period
    to a fixed value — a simulation-only adapter for replaying a specific
    past period through nodes that otherwise always ask for \"current\"."""
    def __init__(self, delegate, pinned_period):
        self.delegate = delegate
        self.pinned_period = pinned_period
    def get_project_finances(self, project_id, reporting_period):
        return self.delegate.get_project_finances(project_id, reporting_period)
    def get_portfolio_finances(self, reporting_period):
        return self.delegate.get_portfolio_finances(reporting_period)
    def get_latest_reporting_period(self, project_id):
        return self.pinned_period

# Isolated store for this section — FileMemoryStore returns snapshots sorted
# by snapshot_date, not insertion order, so reusing Part A's store (which
# already holds September-dated entries) would make these February-dated
# replay entries sort BEFORE them, not after. Exact-date lookup below is
# the robust fix; an isolated store just keeps this section's output easy
# to read on its own.
replay_dir = PROJECT_ROOT / "data/snapshots_replay"
shutil.rmtree(replay_dir, ignore_errors=True)
replay_store = FileMemoryStore(base_dir=replay_dir)

base_fin_source = CSVFinancialDataSource()
week1_deps = NodeDeps(jira_client=build_default_jira_client(), financial_source=PeriodPinnedFinancialSource(base_fin_source, "2026-02"), memory_store=replay_store, mapping=mapping)
week2_deps = NodeDeps(jira_client=build_default_jira_client(), financial_source=PeriodPinnedFinancialSource(base_fin_source, "2026-03"), memory_store=replay_store, mapping=mapping)

graph_week1 = build_graph(week1_deps)
graph_week2 = build_graph(week2_deps)

for g, requested_at in [(graph_week1, datetime(2026, 2, 5, tzinfo=timezone.utc)), (graph_week2, datetime(2026, 2, 20, tzinfo=timezone.utc))]:
    g.invoke({"user_question": "Phoenix and Titan status", "request_id": str(requested_at), "requested_at": requested_at, "project_filter": ["PROJECT-10001", "PROJECT-10004"]})

replay_phx = replay_store.get_snapshots(scope, "PROJECT-10001")
replay_titan = replay_store.get_snapshots(scope, "PROJECT-10004")

# Looked up by exact date, not list position — robust regardless of sort order.
phx_w1 = next(s for s in replay_phx if s.snapshot_date == date(2026, 2, 5))
phx_w2 = next(s for s in replay_phx if s.snapshot_date == date(2026, 2, 20))
titan_w1 = next(s for s in replay_titan if s.snapshot_date == date(2026, 2, 5))
titan_w2 = next(s for s in replay_titan if s.snapshot_date == date(2026, 2, 20))

print(f"PHX:   {phx_w1.sprint_completion_pct}% -> {phx_w2.sprint_completion_pct}%   trend={trend_engine.classify_trend(phx_w1, phx_w2).value}")
print(f"TITAN: {titan_w1.sprint_completion_pct}% -> {titan_w2.sprint_completion_pct}%   trend={trend_engine.classify_trend(titan_w1, titan_w2).value}")

{"request_id": "2026-02-05 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.051204+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "2026-02-05 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.051394+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.0}
{"request_id": "2026-02-05 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.051800+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "2026-02-05 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.067629+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 110, "sprint_count": 6, "partial_failure": false}
{"request_id": "2026-02-05 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.067707+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 15.87}
{"request_id": "2026-02-05 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.068259+00:00", "event": "graph_node_start", "node": "validate_delivery_dat

{"request_id": "2026-02-05 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.078957+00:00", "event": "graph_node_end", "node": "validate_financial_data", "latency_ms": 9.59}
{"request_id": "2026-02-05 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.079704+00:00", "event": "graph_node_start", "node": "unify_projects"}
{"request_id": "2026-02-05 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.082402+00:00", "event": "graph_node_end", "node": "unify_projects", "latency_ms": 2.66}
{"request_id": "2026-02-05 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.082916+00:00", "event": "graph_node_start", "node": "retrieve_historical_memory"}
{"request_id": "2026-02-05 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.083066+00:00", "event": "memory_retrieved", "project_count": 2, "snapshot_count": 0}
{"request_id": "2026-02-05 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.083104+00:00", "event": "graph_node_end", "node": "retrieve_historical_memory", "latency_ms": 0.15}
{"request_

Same result as notebook 09 (IMPROVING / DETERIORATING) — this time produced by two full graph invocations, not by calling `snapshot_builder`/`risk_engine` directly.

## Part C: Section 12 through the graph — three consecutive weekly runs

In [4]:
shutil.rmtree(NOTEBOOK_SNAPSHOT_DIR, ignore_errors=True)  # isolate this section's history from Parts A/B
clean_store = FileMemoryStore(base_dir=NOTEBOOK_SNAPSHOT_DIR)
clean_deps = NodeDeps(jira_client=build_default_jira_client(), financial_source=CSVFinancialDataSource(), memory_store=clean_store, mapping=mapping)
clean_graph = build_graph(clean_deps)

for requested_at in [datetime(2026, 9, 1, tzinfo=timezone.utc), datetime(2026, 9, 8, tzinfo=timezone.utc), datetime(2026, 9, 15, tzinfo=timezone.utc)]:
    clean_graph.invoke({"user_question": "Phoenix status", "request_id": str(requested_at), "requested_at": requested_at, "project_filter": ["PROJECT-10001"]})

phx_snaps = clean_store.get_snapshots(scope, "PROJECT-10001")
print(f"{len(phx_snaps)} snapshots persisted\n")

persistent = trend_engine.find_persistent_blockers(phx_snaps)
print(f"Persistent blockers (>=2 consecutive snapshots): {persistent}")
for issue_key in persistent:
    print(f"  {issue_key}: sprint_count_blocked={trend_engine.compute_sprint_count_blocked(issue_key, phx_snaps)}")

{"request_id": "2026-09-01 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.217280+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "2026-09-01 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.217469+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.0}
{"request_id": "2026-09-01 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.217871+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "2026-09-01 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.220310+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 37, "sprint_count": 3, "partial_failure": false}
{"request_id": "2026-09-01 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.220361+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 2.45}
{"request_id": "2026-09-01 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.220801+00:00", "event": "graph_node_start", "node": "validate_delivery_data"

{"request_id": "2026-09-15 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.270079+00:00", "event": "graph_node_end", "node": "validate_financial_data", "latency_ms": 5.09}
{"request_id": "2026-09-15 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.270730+00:00", "event": "graph_node_start", "node": "unify_projects"}
{"request_id": "2026-09-15 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.271700+00:00", "event": "graph_node_end", "node": "unify_projects", "latency_ms": 0.93}
{"request_id": "2026-09-15 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.272077+00:00", "event": "graph_node_start", "node": "retrieve_historical_memory"}
{"request_id": "2026-09-15 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.272505+00:00", "event": "memory_retrieved", "project_count": 1, "snapshot_count": 2}
{"request_id": "2026-09-15 00:00:00+00:00", "timestamp": "2026-09-03T03:22:59.272549+00:00", "event": "graph_node_end", "node": "retrieve_historical_memory", "latency_ms": 0.44}
{"request_

## Retrieving history back through the graph itself

A fourth run's `retrieve_historical_memory` node should see all 3 prior snapshots — proving the loop closes end to end, not just that `FileMemoryStore` works in isolation (already covered by `tests/test_memory_store.py`).

In [5]:
fourth_run = clean_graph.invoke({
    "user_question": "Phoenix status",
    "request_id": "req-4",
    "requested_at": datetime(2026, 9, 22, tzinfo=timezone.utc),
    "project_filter": ["PROJECT-10001"],
})
print(f"historical_context['PROJECT-10001'] has {len(fourth_run['historical_context']['PROJECT-10001'])} prior snapshots")

{"request_id": "req-4", "timestamp": "2026-09-03T03:22:59.299075+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "req-4", "timestamp": "2026-09-03T03:22:59.299878+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.0}
{"request_id": "req-4", "timestamp": "2026-09-03T03:22:59.301428+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "req-4", "timestamp": "2026-09-03T03:22:59.308471+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 37, "sprint_count": 3, "partial_failure": false}
{"request_id": "req-4", "timestamp": "2026-09-03T03:22:59.308554+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 6.95}
{"request_id": "req-4", "timestamp": "2026-09-03T03:22:59.309031+00:00", "event": "graph_node_start", "node": "validate_delivery_data"}
{"request_id": "req-4", "timestamp": "2026-09-03T03:22:59.309118+00:00", "event": "graph_node_end", "node": "validate_

## Validation checks

- [x] Running the live graph forward with unchanged source data correctly yields `STABLE`, not a fabricated trend
- [x] Real historical improvement/deterioration (PHX/TITAN Sprint 2->3) reproduces notebook 09's result through full graph invocations, not direct service calls
- [x] Section 12 persistence (`sprint_count_blocked`, `find_persistent_blockers`) works from graph-persisted history exactly as it did from hand-built snapshots in notebook 09
- [x] A 4th graph invocation's `retrieve_historical_memory` node sees all 3 prior runs' snapshots — the full read/write loop closes through the graph, not just through `FileMemoryStore` directly

## Testing

`tests/test_graph.py::TestFullGraph::test_second_run_finds_history_from_the_first` is the unit-level version of Part C's closing-the-loop check.

## Next step

Phase 9: ELT report generation — replacing `generate_response`'s deterministic template (Phase 8) with a real LLM call over the same `calculated_metrics`/`risks`/`historical_context` inputs, producing Section 15's full executive report format.